In [1]:
import sqlite3
import pandas as pd

# 1. Definieer de bestandsnamen
db_naam = 'BikeToDriveDWH.db'
sql_script = 'DWHschema.sql'

# 2. Maak verbinding met de SQLite database (maakt het bestand aan als het niet bestaat)
conn = sqlite3.connect(db_naam)
cursor = conn.cursor()

# 3. Lees het SQL-script en voer het uit
print("Data Warehouse aanmaken...")
with open(sql_script, 'r') as file:
    sql_queries = file.read()
    cursor.executescript(sql_queries)

print("Data Warehouse aangemaakt!")


conn.commit()
conn.close()

Data Warehouse aanmaken...
Data Warehouse aangemaakt!


In [2]:
import sqlite3
import pandas as pd

conn_sdm = sqlite3.connect('BikeToDrive.db')
conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
cursor_dwh = conn_dwh.cursor()

# ── 1. CREATE MAPPING TABLE (once) ───────────────────────────────────────────
cursor_dwh.executescript("""
    CREATE TABLE IF NOT EXISTS DWH_KeyMapping (
        mapping_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        dwh_table       TEXT    NOT NULL,
        source_table    TEXT    NOT NULL,
        source_key      INTEGER NOT NULL,
        dwh_surrogate   INTEGER NOT NULL,
        UNIQUE (dwh_table, source_table, source_key)
    );
""")
conn_dwh.commit()
print("✅ Mapping table ready.")


# ── 2. HELPER: look up surrogate key for a source row ────────────────────────
def get_surrogate(cursor, dwh_table, source_table, source_key):
    cursor.execute("""
        SELECT dwh_surrogate FROM DWH_KeyMapping
        WHERE dwh_table = ? AND source_table = ? AND source_key = ?
    """, (dwh_table, source_table, source_key))
    row = cursor.fetchone()
    return row[0] if row else None

def register_mapping(cursor, dwh_table, source_table, source_key, dwh_surrogate):
    cursor.execute("""
        INSERT OR IGNORE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate)
        VALUES (?, ?, ?, ?)
    """, (dwh_table, source_table, source_key, dwh_surrogate))


# ── 3. HELPER: find missing source rows ──────────────────────────────────────
def get_missing(source_conn, source_table, source_key,
                dwh_conn, dwh_table, source_table_name_in_mapping):
    df_source = pd.read_sql_query(f"SELECT * FROM {source_table}", source_conn)
    df_mapped = pd.read_sql_query("""
        SELECT source_key FROM DWH_KeyMapping
        WHERE dwh_table = ? AND source_table = ?
    """, dwh_conn, params=(dwh_table, source_table_name_in_mapping))

    already_loaded = set(df_mapped['source_key']) if not df_mapped.empty else set()
    missing = df_source[~df_source[source_key].isin(already_loaded)]
    return missing


# ── 4. DETECT MISSING ROWS ───────────────────────────────────────────────────
tables_to_check = [
    # (source_table, source_key, dwh_table, label)
    ('Klant',              'klantnr',             'DIM_Klant',      'DIM_Klant'),
    ('Monteur',            'monteurnr',           'DIM_Medewerker', 'DIM_Medewerker'),
    ('Fiets',              'fietsnr',             'DIM_Product',    'DIM_Product (Fiets)'),
    ('Accessoire',         'accessoirenr',        'DIM_Product',    'DIM_Product (Accessoire)'),
    ('Leverancier',        'leveranciernr',       'DIM_Partner',    'DIM_Partner (Leverancier)'),
    ('Fabrikant',          'fabrikantnr',         'DIM_Partner',    'DIM_Partner (Fabrikant)'),
    ('Fiets_Verkoop',      'fiets_verkoopnr',     'FACT_Verkoop',   'FACT_Verkoop (Fiets)'),
    ('Accessoire_Verkoop', 'accessoire_verkoopnr','FACT_Verkoop',   'FACT_Verkoop (Accessoire)'),
    ('Fiets_Inkoop',       'inkoopnr',            'FACT_Inkoop',    'FACT_Inkoop (Fiets)'),
    ('Accessoire_Inkoop',  'inkoopnr',            'FACT_Inkoop',    'FACT_Inkoop (Accessoire)'),
    ('Onderhoud',          'onderhoudnr',         'FACT_Onderhoud', 'FACT_Onderhoud'),
]

missing_report = {}
for source_table, source_key, dwh_table, label in tables_to_check:
    missing = get_missing(conn_sdm, source_table, source_key,
                          conn_dwh, dwh_table, source_table)
    missing_report[(source_table, dwh_table)] = missing
    print(f"{label:<35} → {len(missing):>4} missing rows")

conn_sdm.close()
conn_dwh.close()

✅ Mapping table ready.
DIM_Klant                           →   45 missing rows
DIM_Medewerker                      →   16 missing rows
DIM_Product (Fiets)                 →  105 missing rows
DIM_Product (Accessoire)            →   13 missing rows
DIM_Partner (Leverancier)           →   10 missing rows
DIM_Partner (Fabrikant)             →   31 missing rows
FACT_Verkoop (Fiets)                →  151 missing rows
FACT_Verkoop (Accessoire)           →  100 missing rows
FACT_Inkoop (Fiets)                 →  100 missing rows
FACT_Inkoop (Accessoire)            →   50 missing rows
FACT_Onderhoud                      →   50 missing rows


In [3]:
import sqlite3

# Close any lingering connections by forcing a new connection and immediately closing it
# This won't help directly, but the real fix is restarting the kernel first.

# After restarting kernel, run this cell to verify the DB is accessible:
try:
    conn_test = sqlite3.connect('BikeToDriveDWH.db')
    conn_test.execute("SELECT 1")
    conn_test.close()
    print("✅ Database is vrij en toegankelijk.")
except Exception as e:
    print(f"❌ Database nog steeds geblokkeerd: {e}")

✅ Database is vrij en toegankelijk.


Cell 1

In [ ]:
import sqlite3
import pandas as pd
from datetime import date

conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
cursor_dwh = conn_dwh.cursor()

cursor_dwh.executescript("""
    -- ── DIM_Klant: rebuild with surrogate PK ──────────────────────────────
    ALTER TABLE DIM_Klant RENAME TO DIM_Klant_old;

    CREATE TABLE DIM_Klant (
        surrogate_id        INTEGER PRIMARY KEY AUTOINCREMENT,
        klantnr             INTEGER NOT NULL,
        naam                TEXT    NOT NULL,
        adres               TEXT,
        woonplaats          TEXT,
        geslacht            TEXT,
        geboortedatum       TEXT,
        leeftijdscategorie  TEXT,
        geldig_van          TEXT    NOT NULL DEFAULT '1900-01-01',
        geldig_tot          TEXT    NOT NULL DEFAULT '9999-12-31',
        is_actief           INTEGER NOT NULL DEFAULT 1
    );

    INSERT INTO DIM_Klant (klantnr, naam, adres, woonplaats, geslacht,
                           geboortedatum, leeftijdscategorie,
                           geldig_van, geldig_tot, is_actief)
    SELECT klantnr, naam, adres, woonplaats, geslacht,
           geboortedatum, leeftijdscategorie,
           '1900-01-01', '9999-12-31', 1
    FROM DIM_Klant_old;

    DROP TABLE DIM_Klant_old;

    -- ── DIM_Product: rebuild with surrogate PK ─────────────────────────────
    ALTER TABLE DIM_Product RENAME TO DIM_Product_old;

    CREATE TABLE DIM_Product (
        surrogate_id    INTEGER PRIMARY KEY AUTOINCREMENT,
        productnr       INTEGER NOT NULL,
        type            TEXT    NOT NULL,
        naam            TEXT    NOT NULL,
        producttype     TEXT    NOT NULL CHECK (producttype IN ('Fiets', 'Accessoire')),
        standaardprijs  REAL    NOT NULL,
        inkoopprijs     REAL    NOT NULL,
        prijsklasse     TEXT    NOT NULL CHECK (prijsklasse IN ('Low', 'Mid', 'High')),
        kleur           TEXT,
        merk            TEXT,
        geldig_van      TEXT    NOT NULL DEFAULT '1900-01-01',
        geldig_tot      TEXT    NOT NULL DEFAULT '9999-12-31',
        is_actief       INTEGER NOT NULL DEFAULT 1
    );

    INSERT INTO DIM_Product (productnr, type, naam, producttype, standaardprijs,
                              inkoopprijs, prijsklasse, kleur, merk,
                              geldig_van, geldig_tot, is_actief)
    SELECT productnr, type, naam, producttype, standaardprijs,
           inkoopprijs, prijsklasse, kleur, merk,
           '1900-01-01', '9999-12-31', 1
    FROM DIM_Product_old;

    DROP TABLE DIM_Product_old;

    -- ── Mapping table ──────────────────────────────────────────────────────
    CREATE TABLE IF NOT EXISTS DWH_KeyMapping (
        mapping_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        dwh_table       TEXT    NOT NULL,
        source_table    TEXT    NOT NULL,
        source_key      INTEGER NOT NULL,
        dwh_surrogate   INTEGER NOT NULL,
        UNIQUE (dwh_table, source_table, source_key)
    );

    -- CRITICAL FIX: Seed the mapping table so rebuilt rows aren't treated as
    -- new inserts on the next ETL run (which would cause duplicates).
    INSERT OR IGNORE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate)
    SELECT 'DIM_Klant', 'Klant', klantnr, surrogate_id FROM DIM_Klant;

    -- producttype ('Fiets'/'Accessoire') matches the source table names, so this
    -- correctly seeds both product streams in one statement.
    INSERT OR IGNORE INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate)
    SELECT 'DIM_Product', producttype, productnr, surrogate_id FROM DIM_Product;
""")

conn_dwh.commit()
conn_dwh.close()
print("✅ Schema klaar: surrogate PKs aangemaakt, mapping table aangemaakt en geseed.")


Cell 2

In [5]:
conn_sdm = sqlite3.connect('BikeToDrive.db')
conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
cursor_dwh = conn_dwh.cursor()
today = str(date.today())

df_klant = pd.read_sql_query("SELECT * FROM Klant", conn_sdm)
df_mapped = pd.read_sql_query("""
    SELECT source_key, dwh_surrogate FROM DWH_KeyMapping
    WHERE dwh_table='DIM_Klant' AND source_table='Klant'
""", conn_dwh)
already_loaded = set(df_mapped['source_key'])

scd2_cols = ['naam', 'adres', 'woonplaats', 'geslacht', 'geboortedatum']
inserted = updated = 0

for _, row in df_klant.iterrows():
    klantnr = row['klantnr']
    geboortedatum = str(row['geboortedatum']) if pd.notna(row.get('geboortedatum')) else None

    if klantnr not in already_loaded:
        # ── NEW: insert fresh active row ──────────────────────────────────
        cursor_dwh.execute("""
            INSERT INTO DIM_Klant
                (klantnr, naam, adres, woonplaats, geslacht, geboortedatum,
                 leeftijdscategorie, geldig_van, geldig_tot, is_actief)
            VALUES (?,?,?,?,?,?,?,?,?,1)
        """, (klantnr, row['naam'], row.get('adres'), row.get('woonplaats'),
              row.get('geslacht'), geboortedatum, None, today, '9999-12-31'))

        surrogate = cursor_dwh.lastrowid
        cursor_dwh.execute("""
            INSERT OR IGNORE INTO DWH_KeyMapping
                (dwh_table, source_table, source_key, dwh_surrogate)
            VALUES ('DIM_Klant','Klant',?,?)
        """, (klantnr, surrogate))
        inserted += 1

    else:
        # ── EXISTING: check for changes (SCD2) ────────────────────────────
        surrogate = df_mapped.loc[df_mapped['source_key'] == klantnr, 'dwh_surrogate'].values[0]
        df_current = pd.read_sql_query("""
            SELECT * FROM DIM_Klant WHERE klantnr=? AND is_actief=1
        """, conn_dwh, params=(klantnr,))

        if df_current.empty:
            continue

        current = df_current.iloc[0]
        changed = any(
            str(row.get(col, '')) != str(current.get(col, ''))
            for col in scd2_cols
        )

        if changed:
            # Expire old row
            cursor_dwh.execute("""
                UPDATE DIM_Klant SET geldig_tot=?, is_actief=0
                WHERE klantnr=? AND is_actief=1
            """, (today, klantnr))

            # Insert new version
            cursor_dwh.execute("""
                INSERT INTO DIM_Klant
                    (klantnr, naam, adres, woonplaats, geslacht, geboortedatum,
                     leeftijdscategorie, geldig_van, geldig_tot, is_actief)
                VALUES (?,?,?,?,?,?,?,?,?,1)
            """, (klantnr, row['naam'], row.get('adres'), row.get('woonplaats'),
                  row.get('geslacht'), geboortedatum, None, today, '9999-12-31'))

            new_surrogate = cursor_dwh.lastrowid
            cursor_dwh.execute("""
                UPDATE DWH_KeyMapping SET dwh_surrogate=?
                WHERE dwh_table='DIM_Klant' AND source_table='Klant' AND source_key=?
            """, (new_surrogate, klantnr))
            updated += 1

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()
print(f"✅ DIM_Klant → {inserted} ingevoegd, {updated} geüpdatet (SCD2)")

✅ DIM_Klant → 45 ingevoegd, 0 geüpdatet (SCD2)


Cell 3

In [ ]:
conn_sdm = sqlite3.connect('BikeToDrive.db')
conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
cursor_dwh = conn_dwh.cursor()
today = str(date.today())

def prijsklasse(prijs):
    if prijs < 50:    return 'Low'
    elif prijs < 150: return 'Mid'
    else:             return 'High'

def load_dim_product(df_source, source_table, productnr_col, producttype,
                     naam_col, type_col, prijs_col, inkoop_col,
                     kleur_col=None, merk_col=None):

    df_mapped = pd.read_sql_query(f"""
        SELECT source_key, dwh_surrogate FROM DWH_KeyMapping
        WHERE dwh_table='DIM_Product' AND source_table='{source_table}'
    """, conn_dwh)
    already_loaded = set(df_mapped['source_key'])

    inserted = updated = 0

    for _, row in df_source.iterrows():
        src_key = row.iloc[0]
        kleur   = row.get(kleur_col) if kleur_col else None
        merk    = row.get(merk_col)  if merk_col  else None
        pk      = prijsklasse(row[prijs_col])

        if src_key not in already_loaded:
            # ── NEW: insert fresh active row ──────────────────────────────
            cursor_dwh.execute("""
                INSERT INTO DIM_Product
                    (productnr, type, naam, producttype, standaardprijs, inkoopprijs,
                     prijsklasse, kleur, merk, geldig_van, geldig_tot, is_actief)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,1) 
            """, (row[productnr_col], row[type_col], row[naam_col], producttype,
                  row[prijs_col], row[inkoop_col],
                  pk, kleur, merk, today, '9999-12-31'))

            surrogate = cursor_dwh.lastrowid
            cursor_dwh.execute("""
                INSERT OR IGNORE INTO DWH_KeyMapping
                    (dwh_table, source_table, source_key, dwh_surrogate)
                VALUES ('DIM_Product',?,?,?)
            """, (source_table, src_key, surrogate))
            inserted += 1

        else:
            # ── EXISTING: check for changes (SCD2) ────────────────────────
            surrogate = df_mapped.loc[
                df_mapped['source_key'] == src_key, 'dwh_surrogate'
            ].values[0]

            df_current = pd.read_sql_query("""
                SELECT * FROM DIM_Product WHERE surrogate_id=? AND is_actief=1
            """, conn_dwh, params=(int(surrogate),))

            if df_current.empty:
                continue

            current = df_current.iloc[0]

            # FIX: explicitly map source column names to DWH column names so the
            # comparison doesn't fire on mismatched names (e.g. 'soort' vs 'type').
            col_mapping = {
                naam_col:   'naam',
                type_col:   'type',
                prijs_col:  'standaardprijs',
                inkoop_col: 'inkoopprijs',
            }
            if kleur_col: col_mapping[kleur_col] = 'kleur'
            if merk_col:  col_mapping[merk_col]  = 'merk'

            changed = any(
                str(row.get(src_col, '')) != str(current.get(dwh_col, ''))
                for src_col, dwh_col in col_mapping.items()
            )

            if changed:
                # Expire old row
                cursor_dwh.execute("""
                    UPDATE DIM_Product SET geldig_tot=?, is_actief=0
                    WHERE surrogate_id=? AND is_actief=1
                """, (today, int(surrogate)))

                # Insert new version
                cursor_dwh.execute("""
                    INSERT INTO DIM_Product
                        (productnr, type, naam, producttype, standaardprijs, inkoopprijs,
                         prijsklasse, kleur, merk, geldig_van, geldig_tot, is_actief)
                    VALUES (?,?,?,?,?,?,?,?,?,?,1)
                """, (row[productnr_col], row[type_col], row[naam_col], producttype,
                      row[prijs_col], row[inkoop_col],
                      pk, kleur, merk, today, '9999-12-31'))

                new_surrogate = cursor_dwh.lastrowid
                cursor_dwh.execute("""
                    UPDATE DWH_KeyMapping SET dwh_surrogate=?
                    WHERE dwh_table='DIM_Product' AND source_table=? AND source_key=?
                """, (new_surrogate, source_table, src_key))
                updated += 1

    return inserted, updated

# Fietsen
df_fiets = pd.read_sql_query("SELECT * FROM Fiets", conn_sdm)
ins, upd = load_dim_product(
    df_source=df_fiets, source_table='Fiets', producttype='Fiets',
    productnr_col='fietsnr', naam_col='type', type_col='soort',
    prijs_col='standaardprijs', inkoop_col='inkoopprijs',
    kleur_col='kleur', merk_col='merk'
)
print(f"✅ DIM_Product (Fiets)      → {ins} ingevoegd, {upd} geüpdatet (SCD2)")

# Accessoires
df_acc = pd.read_sql_query("SELECT * FROM Accessoire", conn_sdm)
ins, upd = load_dim_product(
    df_source=df_acc, source_table='Accessoire', producttype='Accessoire',
    productnr_col='accessoirenr', naam_col='naam', type_col='soort',
    prijs_col='standaardprijs', inkoop_col='inkoopprijs'
)
print(f"✅ DIM_Product (Accessoire) → {ins} ingevoegd, {upd} geüpdatet (SCD2)")

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()


Cell 4

In [12]:
import sqlite3
import pandas as pd

conn_sdm = sqlite3.connect('BikeToDrive.db')
conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
cursor_dwh = conn_dwh.cursor()

# ── DIM_Medewerker (SCD1) ─────────────────────────────────────────────────
df_monteur = pd.read_sql_query("SELECT * FROM Monteur", conn_sdm)
df_filiaal = pd.read_sql_query("SELECT * FROM Filiaal", conn_sdm)
df_monteur = df_monteur.merge(
    df_filiaal, left_on='filiaal', right_on='filiaalnr', suffixes=('', '_fil')
)
df_mapped_m = pd.read_sql_query("""
    SELECT source_key FROM DWH_KeyMapping
    WHERE dwh_table='DIM_Medewerker' AND source_table='Monteur'
""", conn_dwh)
already_m = set(df_mapped_m['source_key'])

ins_m = upd_m = 0
for _, row in df_monteur.iterrows():
    monteurnr = row['monteurnr']
    if monteurnr not in already_m:
        # INSERT OR REPLACE handles the case where the row already exists
        # due to a previous partial run
        cursor_dwh.execute("""
            INSERT OR REPLACE INTO DIM_Medewerker
                (monteurID, naam, uurloon, woonplaats, filiaal, regio, provincie)
            VALUES (?,?,?,?,?,?,?)
        """, (monteurnr, row['naam'], row['uurloon'],
              row.get('woonplaats'), row.get('naam_fil'),
              None, row.get('provincie')))
        cursor_dwh.execute("""
            INSERT OR IGNORE INTO DWH_KeyMapping
                (dwh_table, source_table, source_key, dwh_surrogate)
            VALUES ('DIM_Medewerker','Monteur',?,?)
        """, (monteurnr, monteurnr))
        ins_m += 1
    else:
        # SCD1: overwrite in place
        cursor_dwh.execute("""
            UPDATE DIM_Medewerker
            SET naam=?, uurloon=?, woonplaats=?, filiaal=?, provincie=?
            WHERE monteurID=?
        """, (row['naam'], row['uurloon'], row.get('woonplaats'),
              row.get('naam_fil'), row.get('provincie'), monteurnr))
        upd_m += 1

print(f"✅ DIM_Medewerker → {ins_m} ingevoegd, {upd_m} geüpdatet (SCD1)")

# ── DIM_Partner (SCD1) ────────────────────────────────────────────────────
def load_dim_partner(df_source, source_table, partnertype, naam_col, locatie_col):
    df_mapped = pd.read_sql_query(f"""
        SELECT source_key, dwh_surrogate FROM DWH_KeyMapping
        WHERE dwh_table='DIM_Partner' AND source_table='{source_table}'
    """, conn_dwh)
    already = set(df_mapped['source_key'])
    ins = upd = 0

    for _, row in df_source.iterrows():
        src_key = row.iloc[0]
        locatie = row.get(locatie_col) if locatie_col in row.index else None

        if src_key not in already:
            cursor_dwh.execute("""
                INSERT OR IGNORE INTO DIM_Partner (naam, partnertype, locatie)
                VALUES (?,?,?)
            """, (row[naam_col], partnertype, locatie))
            surrogate = cursor_dwh.lastrowid
            cursor_dwh.execute("""
                INSERT OR IGNORE INTO DWH_KeyMapping
                    (dwh_table, source_table, source_key, dwh_surrogate)
                VALUES ('DIM_Partner',?,?,?)
            """, (source_table, src_key, surrogate))
            ins += 1
        else:
            surrogate = df_mapped.loc[
                df_mapped['source_key'] == src_key, 'dwh_surrogate'
            ].values[0]
            cursor_dwh.execute("""
                UPDATE DIM_Partner SET naam=?, locatie=? WHERE partnernr=?
            """, (row[naam_col], locatie, int(surrogate)))
            upd += 1
    return ins, upd

df_lev = pd.read_sql_query("SELECT * FROM Leverancier", conn_sdm)
df_fab = pd.read_sql_query("SELECT * FROM Fabrikant",   conn_sdm)

ins, upd = load_dim_partner(df_lev, 'Leverancier', 'Leverancier', 'naam', 'woonplaats')
print(f"✅ DIM_Partner (Leverancier) → {ins} ingevoegd, {upd} geüpdatet (SCD1)")

ins, upd = load_dim_partner(df_fab, 'Fabrikant', 'Fabrikant', 'naam', 'plaats')
print(f"✅ DIM_Partner (Fabrikant)   → {ins} ingevoegd, {upd} geüpdatet (SCD1)")

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()

✅ DIM_Medewerker → 42 ingevoegd, 0 geüpdatet (SCD1)
✅ DIM_Partner (Leverancier) → 10 ingevoegd, 0 geüpdatet (SCD1)
✅ DIM_Partner (Fabrikant)   → 31 ingevoegd, 0 geüpdatet (SCD1)


Cell 5

In [ ]:
import sqlite3
import pandas as pd
from datetime import date

conn_sdm = sqlite3.connect('BikeToDrive.db')
conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
cursor_dwh = conn_dwh.cursor()

# ── Helpers ───────────────────────────────────────────────────────────────
def ensure_datum(datum_str):
    if not datum_str or datum_str == 'None':
        return None
    try:
        d = pd.to_datetime(datum_str)
        datum = str(d.date())
        cursor_dwh.execute("SELECT 1 FROM DIM_Tijd WHERE datum=?", (datum,))
        if not cursor_dwh.fetchone():
            cursor_dwh.execute("""
                INSERT INTO DIM_Tijd (datum, dag, maand, kwartaal, jaar, weekend)
                VALUES (?,?,?,?,?,?)
            """, (datum, d.day, d.month,
                  (d.month - 1) // 3 + 1, d.year,
                  1 if d.weekday() >= 5 else 0))
        return datum
    except Exception:
        return None

def get_surrogate(dwh_table, source_table, source_key):
    cursor_dwh.execute("""
        SELECT dwh_surrogate FROM DWH_KeyMapping
        WHERE dwh_table=? AND source_table=? AND source_key=?
    """, (dwh_table, source_table, int(source_key)))
    row = cursor_dwh.fetchone()
    return row[0] if row else None

def already_in_fact(fact_table, source_table, src_key):
    cursor_dwh.execute("""
        SELECT 1 FROM DWH_KeyMapping
        WHERE dwh_table=? AND source_table=? AND source_key=?
    """, (fact_table, source_table, int(src_key)))
    return cursor_dwh.fetchone() is not None

def register_fact(fact_table, source_table, src_key, fact_id):
    cursor_dwh.execute("""
        INSERT INTO DWH_KeyMapping (dwh_table, source_table, source_key, dwh_surrogate)
        VALUES (?,?,?,?)
    """, (fact_table, source_table, int(src_key), fact_id))

# ── Verify column names before starting ───────────────────────────────────
print("Fiets_Inkoop columns:      ", pd.read_sql_query("SELECT * FROM Fiets_Inkoop LIMIT 1", conn_sdm).columns.tolist())
print("Accessoire_Inkoop columns: ", pd.read_sql_query("SELECT * FROM Accessoire_Inkoop LIMIT 1", conn_sdm).columns.tolist())
print("Fiets_Verkoop columns:     ", pd.read_sql_query("SELECT * FROM Fiets_Verkoop LIMIT 1", conn_sdm).columns.tolist())
print("Accessoire_Verkoop columns:", pd.read_sql_query("SELECT * FROM Accessoire_Verkoop LIMIT 1", conn_sdm).columns.tolist())
print("Onderhoud columns:         ", pd.read_sql_query("SELECT * FROM Onderhoud LIMIT 1", conn_sdm).columns.tolist())
print()

# ── FACT_Verkoop ──────────────────────────────────────────────────────────
ins_fv = ins_av = skipped_fv = skipped_av = 0

df_fv = pd.read_sql_query("SELECT * FROM Fiets_Verkoop", conn_sdm)
for _, row in df_fv.iterrows():
    src_key = row['fiets_verkoopnr']
    if already_in_fact('FACT_Verkoop', 'Fiets_Verkoop', src_key):
        continue
    datum = ensure_datum(str(row['datum']))
    prod  = get_surrogate('DIM_Product',    'Fiets',   row['fiets'])
    klant = get_surrogate('DIM_Klant',      'Klant',   row['klant'])
    medew = get_surrogate('DIM_Medewerker', 'Monteur', row['monteur'])
    if not all([datum, prod, klant, medew]):
        skipped_fv += 1
        continue
    omzet = row['aantal'] * row['verkoopprijs']
    cursor_dwh.execute("""
        INSERT INTO FACT_Verkoop
            (productnr, klantnr, medewerkerID, datum, aantal, verkoopprijs, omzet)
        VALUES (?,?,?,?,?,?,?)
    """, (prod, klant, medew, datum, row['aantal'], row['verkoopprijs'], omzet))
    register_fact('FACT_Verkoop', 'Fiets_Verkoop', src_key, cursor_dwh.lastrowid)
    ins_fv += 1

df_av = pd.read_sql_query("SELECT * FROM Accessoire_Verkoop", conn_sdm)
for _, row in df_av.iterrows():
    src_key = row['accessoire_verkoopnr']
    if already_in_fact('FACT_Verkoop', 'Accessoire_Verkoop', src_key):
        continue
    datum = ensure_datum(str(row['datum']))
    prod  = get_surrogate('DIM_Product',    'Accessoire', row['accessoire'])
    klant = get_surrogate('DIM_Klant',      'Klant',      row['klant'])
    medew = get_surrogate('DIM_Medewerker', 'Monteur',    row['monteur'])
    if not all([datum, prod, klant, medew]):
        skipped_av += 1
        continue
    omzet = row['aantal'] * row['verkoopprijs']
    cursor_dwh.execute("""
        INSERT INTO FACT_Verkoop
            (productnr, klantnr, medewerkerID, datum, aantal, verkoopprijs, omzet)
        VALUES (?,?,?,?,?,?,?)
    """, (prod, klant, medew, datum, row['aantal'], row['verkoopprijs'], omzet))
    register_fact('FACT_Verkoop', 'Accessoire_Verkoop', src_key, cursor_dwh.lastrowid)
    ins_av += 1

print(f"✅ FACT_Verkoop  → {ins_fv} Fiets_Verkoop, {ins_av} Accessoire_Verkoop ingevoegd "
      f"({skipped_fv + skipped_av} overgeslagen)")

# ── FACT_Inkoop ───────────────────────────────────────────────────────────
# schema.sql: Fiets_Inkoop has column 'fietsnr' as FK
#             Accessoire_Inkoop has column 'accessoirenr' as FK
ins_fi = ins_ai = skipped_fi = skipped_ai = 0

df_fi = pd.read_sql_query("SELECT * FROM Fiets_Inkoop", conn_sdm)
df_fiets_all = pd.read_sql_query(
    "SELECT fietsnr, fabrikant, inkoopprijs FROM Fiets", conn_sdm
)

# Detect FK column name for Fiets_Inkoop dynamically
fiets_fk = 'fietsnr' if 'fietsnr' in df_fi.columns else 'fiets'

for _, row in df_fi.iterrows():
    src_key   = row['inkoopnr']
    fiets_key = row[fiets_fk]
    if already_in_fact('FACT_Inkoop', 'Fiets_Inkoop', src_key):
        continue
    datum = ensure_datum(
        f"{int(row['inkoopjaar'])}-{int(row['inkoopmaand']):02d}-01"
    )
    prod = get_surrogate('DIM_Product', 'Fiets', fiets_key)

    fiets_info = df_fiets_all[df_fiets_all['fietsnr'] == fiets_key]
    if fiets_info.empty or not prod or not datum:
        skipped_fi += 1
        continue
    partner = get_surrogate('DIM_Partner', 'Fabrikant',
                            fiets_info.iloc[0]['fabrikant'])
    if not partner:
        skipped_fi += 1
        continue
    inkoopwaarde = row['aantal'] * fiets_info.iloc[0]['inkoopprijs']
    cursor_dwh.execute("""
        INSERT INTO FACT_Inkoop (productnr, partnernr, datum, aantal, inkoopwaarde)
        VALUES (?,?,?,?,?)
    """, (prod, partner, datum, row['aantal'], inkoopwaarde))
    register_fact('FACT_Inkoop', 'Fiets_Inkoop', src_key, cursor_dwh.lastrowid)
    ins_fi += 1

df_ai = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", conn_sdm)
df_acc_all = pd.read_sql_query(
    "SELECT accessoirenr, leverancier, inkoopprijs FROM Accessoire", conn_sdm
)

# Detect FK column name for Accessoire_Inkoop dynamically
acc_fk = 'accessoirenr' if 'accessoirenr' in df_ai.columns else 'accessoire'

for _, row in df_ai.iterrows():
    src_key  = row['inkoopnr']
    acc_key  = row[acc_fk]
    if already_in_fact('FACT_Inkoop', 'Accessoire_Inkoop', src_key):
        continue
    datum = ensure_datum(
        f"{int(row['inkoopjaar'])}-{int(row['inkoopmaand']):02d}-01"
    )
    prod = get_surrogate('DIM_Product', 'Accessoire', acc_key)

    acc_info = df_acc_all[df_acc_all['accessoirenr'] == acc_key]
    if acc_info.empty or not prod or not datum:
        skipped_ai += 1
        continue
    partner = get_surrogate('DIM_Partner', 'Leverancier',
                            acc_info.iloc[0]['leverancier'])
    if not partner:
        skipped_ai += 1
        continue
    inkoopwaarde = row['aantal'] * acc_info.iloc[0]['inkoopprijs']
    cursor_dwh.execute("""
        INSERT INTO FACT_Inkoop (productnr, partnernr, datum, aantal, inkoopwaarde)
        VALUES (?,?,?,?,?)
    """, (prod, partner, datum, row['aantal'], inkoopwaarde))
    register_fact('FACT_Inkoop', 'Accessoire_Inkoop', src_key, cursor_dwh.lastrowid)
    ins_ai += 1

print(f"✅ FACT_Inkoop   → {ins_fi} Fiets_Inkoop, {ins_ai} Accessoire_Inkoop ingevoegd "
      f"({skipped_fi + skipped_ai} overgeslagen)")

# ── FACT_Onderhoud ────────────────────────────────────────────────────────
ins_oh = skipped_oh = 0
df_oh = pd.read_sql_query("SELECT * FROM Onderhoud", conn_sdm)
df_fv_klant = pd.read_sql_query("SELECT fiets, klant FROM Fiets_Verkoop", conn_sdm)

# Detect FK column name for Onderhoud dynamically
fiets_fk_oh  = 'fietsnr'   if 'fietsnr'   in df_oh.columns else 'fiets'
monteur_fk   = 'monteurnr' if 'monteurnr' in df_oh.columns else 'monteur'

for _, row in df_oh.iterrows():
    src_key = row['onderhoudnr']
    if already_in_fact('FACT_Onderhoud', 'Onderhoud', src_key):
        continue
    datum = ensure_datum(str(row['datum']))
    medew = get_surrogate('DIM_Medewerker', 'Monteur', row[monteur_fk])

    # Skip only if essential fields are missing
    if not datum or not medew:
        skipped_oh += 1
        continue

    # FIX: allow pre-sale maintenance (bike not yet sold) — inserts NULL for klantnr.
    # klantnr is nullable in FACT_Onderhoud for exactly this reason.
    klant_match = df_fv_klant[df_fv_klant['fiets'] == row[fiets_fk_oh]]
    if not klant_match.empty:
        klant = get_surrogate('DIM_Klant', 'Klant', klant_match.iloc[0]['klant'])
    else:
        klant = None  # pre-sale maintenance: no customer yet

    cursor_dwh.execute("""
        INSERT INTO FACT_Onderhoud (medewerkerID, klantnr, datum, begintijd, eindtijd)
        VALUES (?,?,?,?,?)
    """, (medew, klant, datum, str(row['starttijd']), str(row['eindtijd'])))
    register_fact('FACT_Onderhoud', 'Onderhoud', src_key, cursor_dwh.lastrowid)
    ins_oh += 1

print(f"✅ FACT_Onderhoud → {ins_oh} ingevoegd ({skipped_oh} overgeslagen)")

conn_dwh.commit()
conn_sdm.close()
conn_dwh.close()
print("\n✅ ETL volledig afgerond.")


In [14]:
import sqlite3
conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
import pandas as pd

print("SCD1 Check (Monteur 1 is overschreven):")
display(pd.read_sql_query("SELECT * FROM DIM_Medewerker WHERE monteurID = 1", conn_dwh))

print("\nSCD2 Check (Klant 1 heeft nu twee rijen: een oude en een actieve):")
display(pd.read_sql_query("SELECT * FROM DIM_Klant WHERE klantnr = 1", conn_dwh))

print("\nFact Check (De nieuwste verkoop verwijst naar de NIEUWE surrogate key van Klant 1):")
display(pd.read_sql_query("SELECT * FROM FACT_Verkoop ORDER BY verkoop_id DESC LIMIT 1", conn_dwh))

conn_dwh.close()

SCD1 Check (Monteur 1 is overschreven):


,monteurID,naam,uurloon,woonplaats,filiaal,regio,provincie
0,1,Tom van Dijk,19.5,Test Woonplaats SCD1,BikeWorld Amsterdam,None,Noord-Holland



SCD2 Check (Klant 1 heeft nu twee rijen: een oude en een actieve):


,surrogate_id,klantnr,naam,adres,woonplaats,geslacht,geboortedatum,leeftijdscategorie,geldig_van,geldig_tot,is_actief
0,1,1,Jan Jansen,Nieuw Adres SCD2 Test,Amsterdam,M,1985-03-22,None,2026-04-02,9999-12-31,1
1,21,1,Jan Jansen,Nieuw Adres SCD2 Test,Amsterdam,M,1985-03-22,None,2026-04-02,9999-12-31,1



Fact Check (De nieuwste verkoop verwijst naar de NIEUWE surrogate key van Klant 1):


,verkoop_id,productnr,klantnr,medewerkerID,datum,aantal,verkoopprijs,omzet
0,251,110,18,10,2024-10-27,4,12.67,50.68


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SCD Type 2 Demo
#
# Simulates a real-world address change in the source system and shows
# how the ETL creates a new DIM_Klant version while preserving the old one.
#
# Run this cell as many times as you like — each run picks a new city,
# so you'll see the history grow one row at a time.
# ══════════════════════════════════════════════════════════════════════════════

import sqlite3
import pandas as pd
from datetime import date
import itertools

DEMO_KLANTNR = 2  # Sophie de Boer — feels free to swap to any valid klantnr

# Cycle through cities so each run produces a visible change
CITIES = itertools.cycle(['Rotterdam', 'Utrecht', 'Groningen', 'Eindhoven', 'Maastricht'])

conn_sdm = sqlite3.connect('BikeToDrive.db')
conn_dwh = sqlite3.connect('BikeToDriveDWH.db')
cursor_sdm = conn_sdm.cursor()
cursor_dwh = conn_dwh.cursor()
today = str(date.today())

# ── Step 1: snapshot current DIM state ───────────────────────────────────────
print(f"BEFORE  —  DIM_Klant rows for klantnr {DEMO_KLANTNR}")
df_before = pd.read_sql_query("""
    SELECT surrogate_id, klantnr, naam, woonplaats,
           geldig_van, geldig_tot, is_actief
    FROM DIM_Klant WHERE klantnr=? ORDER BY surrogate_id
""", conn_dwh, params=(DEMO_KLANTNR,))
display(df_before)

# ── Step 2: pick the next city that differs from the current active value ─────
current_woonplaats = df_before.loc[df_before['is_actief'] == 1, 'woonplaats'].values
current_woonplaats = current_woonplaats[0] if len(current_woonplaats) else None

new_woonplaats = next(c for c in CITIES if c != current_woonplaats)

cursor_sdm.execute(
    "UPDATE Klant SET woonplaats=? WHERE klantnr=?",
    (new_woonplaats, DEMO_KLANTNR)
)
conn_sdm.commit()
print(f"\n→  Source bijgewerkt:  woonplaats  '{current_woonplaats}'  →  '{new_woonplaats}'")

# ── Step 3: SCD2 ETL for this one customer ────────────────────────────────────
scd2_cols = ['naam', 'adres', 'woonplaats', 'geslacht', 'geboortedatum']

df_mapped = pd.read_sql_query("""
    SELECT source_key, dwh_surrogate FROM DWH_KeyMapping
    WHERE dwh_table='DIM_Klant' AND source_table='Klant'
""", conn_dwh)

src_row = pd.read_sql_query(
    "SELECT * FROM Klant WHERE klantnr=?", conn_sdm, params=(DEMO_KLANTNR,)
).iloc[0]

geboortedatum = str(src_row['geboortedatum']) if pd.notna(src_row.get('geboortedatum')) else None

df_current_active = pd.read_sql_query(
    "SELECT * FROM DIM_Klant WHERE klantnr=? AND is_actief=1",
    conn_dwh, params=(DEMO_KLANTNR,)
)

if df_current_active.empty:
    print("⚠️  Geen actieve rij gevonden voor deze klant — run de dim-laad cel eerst.")
else:
    current = df_current_active.iloc[0]
    changed = any(
        str(src_row.get(col, '')) != str(current.get(col, ''))
        for col in scd2_cols
    )

    if changed:
        # Expire old active row
        cursor_dwh.execute(
            "UPDATE DIM_Klant SET geldig_tot=?, is_actief=0 WHERE klantnr=? AND is_actief=1",
            (today, DEMO_KLANTNR)
        )
        # Insert new version
        cursor_dwh.execute("""
            INSERT INTO DIM_Klant
                (klantnr, naam, adres, woonplaats, geslacht, geboortedatum,
                 leeftijdscategorie, geldig_van, geldig_tot, is_actief)
            VALUES (?,?,?,?,?,?,?,?,?,1)
        """, (DEMO_KLANTNR, src_row['naam'], src_row.get('adres'), src_row.get('woonplaats'),
              src_row.get('geslacht'), geboortedatum, None, today, '9999-12-31'))

        new_surrogate = cursor_dwh.lastrowid
        cursor_dwh.execute("""
            UPDATE DWH_KeyMapping SET dwh_surrogate=?
            WHERE dwh_table='DIM_Klant' AND source_table='Klant' AND source_key=?
        """, (new_surrogate, DEMO_KLANTNR))
        conn_dwh.commit()
        print("✅  Wijziging gedetecteerd — nieuwe SCD2-versie aangemaakt.\n")
    else:
        print("ℹ️  Geen wijziging gedetecteerd (waarde was al hetzelfde in het DWH).\n")

# ── Step 4: show full history ─────────────────────────────────────────────────
print(f"AFTER  —  DIM_Klant rows for klantnr {DEMO_KLANTNR}  (volledig historiek)")
display(pd.read_sql_query("""
    SELECT surrogate_id, klantnr, naam, woonplaats,
           geldig_van, geldig_tot,
           CASE is_actief WHEN 1 THEN '✅ actief' ELSE '🗄 verlopen' END AS status
    FROM DIM_Klant WHERE klantnr=? ORDER BY surrogate_id
""", conn_dwh, params=(DEMO_KLANTNR,)))

conn_sdm.close()
conn_dwh.close()
